# 성능 비교 시각화 v4_improved — 2026-05-26

**담당:** 경이 (kyeongyi)  
**목적:** Simple (TF-IDF + LogReg) vs KcELECTRA v3_2 vs KcELECTRA v4_improved 3-way 비교  
**베이스 모델:** `beomi/kcelectra-base` (KcELECTRA — 이준범 제작)

---

## 왜 Simple과 v3_2 수치를 먼저 구해야 하는가?

3-way 비교는 **동일한 test 세트(1,593건 original)**를 기준으로 합니다.

| 모델 | 학습 데이터 | 테스트 데이터 | 비고 |
|------|-----------|------------|------|
| Simple (TF-IDF + LogReg) | split_v4 **train** (13,210개) | split_v4 **test original** (1,593건) | Step 1에서 새로 실행 |
| KcELECTRA v3_2 | split_v3_1 train (12,759개) | split_v3_1 test (1,593건) = **동일** | 이미 존재 |
| KcELECTRA v4_improved | split_v4 train (13,210개) | split_v4 **test original** (1,593건) | Colab 학습 후 |

> **핵심**: split_v4의 test original 1,593건 = split_v3_1의 test 1,593건 (완전히 동일한 샘플)  
> 따라서 v3_2 JSON은 재사용 가능. Simple만 split_v4 train으로 재학습 필요.

---

## 실행 순서 (단계별)

### Step 1 — 로컬: Simple을 split_v4 데이터로 평가
```bash
cd model/classification
python scripts/evaluate_compare_v4_schoolalimi_20260525.py --no-chart
```
→ 생성: `data/20260525/eval_results_simple_v4_20260525.json`

### Step 2 — Colab: v4_improved 학습
1. `split_v4_schoolalimi_20260525.csv` 업로드 (`/content/`)
2. `15_train_kcelectra_v4_improved_20260526.ipynb` 전체 실행 (~50분)
3. `/content/eval_results_kcelectra_v4_improved_20260526.json` 다운로드
4. 로컬 `data/20260526/` 폴더에 배치

### Step 3 — 로컬: 이 시각화 노트북 실행
- 위 두 단계가 완료된 후 실행

---

## 필요 파일 체크리스트
- [ ] `data/20260525/eval_results_simple_v4_20260525.json` (Step 1 완료 후)
- [ ] `data/20260509/eval_results_kcelectra_v3_2_20260509.json` (이미 존재)
- [ ] `data/20260526/eval_results_kcelectra_v4_improved_20260526.json` (Step 2 완료 후)

## 생성 파일 (`data/20260526/`)
- `compare_macro_f1_v4_improved_20260526.png` — Macro F1 3-way 막대 비교
- `compare_per_class_f1_v4_improved_20260526.png` — 카테고리별 F1 3-way 비교
- `compare_confusion_matrix_v4_improved_20260526.png` — Confusion Matrix 나란히
- `compare_radar_f1_v4_improved_20260526.png` — 레이더 차트
- `compare_version_trend_v4_improved_20260526.png` — 버전별 성능 추이
- `compare_precision_recall_f1_v4_improved_20260526.png` — Precision/Recall/F1 종합

In [ ]:
# 셀 1: 경로 설정 + 데이터 로드
import json
import platform
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# 한글 폰트 설정
if platform.system() == 'Windows':
    matplotlib.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    matplotlib.rc('font', family='AppleGothic')
else:
    try:
        import subprocess
        import matplotlib.font_manager as fm
        subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
        font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
        fm.fontManager.addfont(font_path)
        prop = fm.FontProperties(fname=font_path)
        matplotlib.rc('font', family=prop.get_name())
    except Exception:
        pass
matplotlib.rcParams['axes.unicode_minus'] = False

# 경로 설정 (로컬 / Colab 모두 지원)
_NB   = Path('.').resolve()
_BASE = _NB.parent if _NB.name == 'notebooks' else _NB

DIR_0509 = _BASE / 'data' / '20260509'
DIR_0525 = _BASE / 'data' / '20260525'
DIR_0526 = _BASE / 'data' / '20260526'
OUT      = DIR_0526
OUT.mkdir(parents=True, exist_ok=True)

# Simple: split_v4 train 기반 우선, 없으면 v3_2 train 기반 fallback
_simple_v4  = DIR_0525 / 'eval_results_simple_v4_20260525.json'
_simple_v32 = DIR_0509 / 'eval_results_simple_v3_2_20260509.json'
if _simple_v4.exists():
    SIMPLE_JSON = _simple_v4
    print('[로드] Simple: split_v4 train 기반 (eval_results_simple_v4_20260525.json)')
else:
    SIMPLE_JSON = _simple_v32
    print('[주의] Simple: split_v4 기반 JSON 없음 → v3_2 train 기반 fallback')
    print('       → 정확한 3-way 비교를 위해 Step 1 실행 권장:')
    print('         python scripts/evaluate_compare_v4_schoolalimi_20260525.py --no-chart')

V3_2_JSON = DIR_0509 / 'eval_results_kcelectra_v3_2_20260509.json'
V4I_JSON  = DIR_0526 / 'eval_results_kcelectra_v4_improved_20260526.json'

LABELS = ['일정', '준비물', '제출', '비용', '건강·안전', '기타']

# --- 로드 ---
if not SIMPLE_JSON.exists():
    raise FileNotFoundError(f'{SIMPLE_JSON} 없음 — Step 1 먼저 실행')
if not V3_2_JSON.exists():
    raise FileNotFoundError(f'{V3_2_JSON} 없음 — v3_2 Colab 결과 필요')

with open(SIMPLE_JSON, encoding='utf-8') as f:
    simple = json.load(f)
with open(V3_2_JSON, encoding='utf-8') as f:
    v3_2 = json.load(f)

v4i = None
if V4I_JSON.exists():
    with open(V4I_JSON, encoding='utf-8') as f:
        v4i = json.load(f)
    print(f'[로드] v4_improved 결과 있음')
else:
    print(f'[주의] v4_improved JSON 없음 — v3_2까지만 시각화합니다.')
    print(f'  → Colab에서 15_train_kcelectra_v4_improved_20260526.ipynb 실행 후 재시도')

s_f1   = simple['macro_f1']
v3_f1  = v3_2['macro_f1']
v4i_f1 = v4i['macro_f1'] if v4i else None

print(f'Simple      Macro F1: {s_f1:.4f}')
print(f'v3_2        Macro F1: {v3_f1:.4f}')
if v4i_f1:
    print(f'v4_improved Macro F1: {v4i_f1:.4f}  (v3_2 대비 {v4i_f1-v3_f1:+.4f})')

In [ ]:
# 셀 2: [그래프 1] Macro F1 3-way 막대 비교

if v4i is None:
    models = ['Simple\n(TF-IDF + LogReg)', 'KcELECTRA v3_2\n(beomi)']
    f1s    = [s_f1, v3_f1]
    colors = ['#6baed6', '#e6550d']
else:
    models = ['Simple\n(TF-IDF + LogReg)', 'KcELECTRA v3_2\n(beomi)', 'KcELECTRA v4_improved\n(beomi)']
    f1s    = [s_f1, v3_f1, v4i_f1]
    colors = ['#6baed6', '#e6550d', '#31a354']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models, f1s, color=colors, width=0.4, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=13, fontweight='bold')

# delta 화살표: Simple → v3_2
ax.annotate('', xy=(1, v3_f1 - 0.005), xytext=(0, s_f1 + 0.005),
            arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))
ax.text(0.5, (s_f1 + v3_f1) / 2, f'+{v3_f1-s_f1:.4f}',
        ha='center', va='bottom', fontsize=10, color='#333333',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', edgecolor='#cccccc'))

# delta 화살표: v3_2 → v4_improved
if v4i is not None:
    sign_color = '#2ca02c' if v4i_f1 >= v3_f1 else '#d62728'
    ax.annotate('', xy=(2, v4i_f1 - 0.005), xytext=(1, v3_f1 + 0.005),
                arrowprops=dict(arrowstyle='->', color=sign_color, lw=1.5))
    ax.text(1.5, (v3_f1 + v4i_f1) / 2, f'{v4i_f1-v3_f1:+.4f}',
            ha='center', va='bottom', fontsize=10, color=sign_color,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='lightgreen', edgecolor='#cccccc'))

ax.set_ylim(0, 1.08)
ax.set_ylabel('Macro F1', fontsize=13)
ax.set_title('베이스라인 vs KcELECTRA 성능 비교\n(test 1,593건, 동일 test set)', fontsize=13)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4, linewidth=1)
ax.text(len(models) - 0.45, 0.802, 'F1=0.80', color='gray', fontsize=9, va='bottom')
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
out_path = OUT / 'compare_macro_f1_v4_improved_20260526.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {out_path}')

In [ ]:
# 셀 3: [그래프 2] 카테고리별 F1 비교 (3-way grouped bar)

s_f1s  = [simple['per_class'][lbl]['f1']  for lbl in LABELS]
v3_f1s = [v3_2['per_class'][lbl]['f1']   for lbl in LABELS]

x = np.arange(len(LABELS))

if v4i is None:
    w = 0.35
    fig, ax = plt.subplots(figsize=(11, 6))
    b1 = ax.bar(x - w/2, s_f1s,  w, label='Simple (베이스라인)', color='#6baed6', edgecolor='white')
    b2 = ax.bar(x + w/2, v3_f1s, w, label='KcELECTRA v3_2', color='#e6550d', edgecolor='white')
    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, color='#2171b5')
    for bar in b2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, color='#a63603')
else:
    w = 0.25
    v4i_f1s = [v4i['per_class'][lbl]['f1'] for lbl in LABELS]
    fig, ax = plt.subplots(figsize=(13, 6))
    b1 = ax.bar(x - w,   s_f1s,   w, label='Simple (베이스라인)', color='#6baed6', edgecolor='white')
    b2 = ax.bar(x,       v3_f1s,  w, label='KcELECTRA v3_2',      color='#e6550d', edgecolor='white')
    b3 = ax.bar(x + w,   v4i_f1s, w, label='KcELECTRA v4_improved', color='#31a354', edgecolor='white')
    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.008,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='#2171b5')
    for bar in b2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.008,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='#a63603')
    for bar in b3:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.008,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='#006d2c')

ax.set_xticks(x)
ax.set_xticklabels(LABELS, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('카테고리별 F1 비교: Simple vs v3_2 vs v4_improved\n(beomi/kcelectra-base)', fontsize=13)
ax.legend(fontsize=11)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.3, linewidth=1)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
out_path = OUT / 'compare_per_class_f1_v4_improved_20260526.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {out_path}')

In [ ]:
# 셀 4: [그래프 3] Confusion Matrix 나란히 (v3_2 vs v4_improved)

s_cm   = np.array(simple['confusion_matrix'])
v3_cm  = np.array(v3_2['confusion_matrix'])

if v4i is None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    pairs = [
        (axes[0], s_cm,  f'Simple (TF-IDF + LogReg)\nMacro F1={s_f1:.4f}'),
        (axes[1], v3_cm, f'KcELECTRA v3_2 (beomi)\nMacro F1={v3_f1:.4f}'),
    ]
else:
    v4i_cm = np.array(v4i['confusion_matrix'])
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    pairs = [
        (axes[0], s_cm,   f'Simple (TF-IDF + LogReg)\nMacro F1={s_f1:.4f}'),
        (axes[1], v3_cm,  f'KcELECTRA v3_2 (beomi)\nMacro F1={v3_f1:.4f}'),
        (axes[2], v4i_cm, f'KcELECTRA v4_improved\nMacro F1={v4i_f1:.4f}'),
    ]

for ax, cm_data, title in pairs:
    sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS,
                ax=ax, cbar=False)
    ax.set_xlabel('예측 카테고리', fontsize=10)
    ax.set_ylabel('실제 카테고리', fontsize=10)
    ax.set_title(title, fontsize=11, pad=10)

plt.suptitle('Confusion Matrix 비교 (test 1,593건 — 동일 test set)', fontsize=14, y=1.01)
plt.tight_layout()
out_path = OUT / 'compare_confusion_matrix_v4_improved_20260526.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {out_path}')

In [ ]:
# 셀 5: [그래프 4] 레이더 차트 (3 lines)

angles = np.linspace(0, 2 * np.pi, len(LABELS), endpoint=False).tolist()
angles += angles[:1]

s_vals  = [simple['per_class'][lbl]['f1'] for lbl in LABELS]
s_vals += s_vals[:1]
v3_vals = [v3_2['per_class'][lbl]['f1']   for lbl in LABELS]
v3_vals += v3_vals[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

ax.plot(angles, s_vals,  'o-', linewidth=2, color='#6baed6', label=f'Simple (F1={s_f1:.4f})')
ax.fill(angles, s_vals,  alpha=0.1, color='#6baed6')
ax.plot(angles, v3_vals, 's-', linewidth=2, color='#e6550d', label=f'KcELECTRA v3_2 (F1={v3_f1:.4f})')
ax.fill(angles, v3_vals, alpha=0.1, color='#e6550d')

if v4i is not None:
    v4i_vals  = [v4i['per_class'][lbl]['f1'] for lbl in LABELS]
    v4i_vals += v4i_vals[:1]
    ax.plot(angles, v4i_vals, '^-', linewidth=2, color='#31a354',
            label=f'KcELECTRA v4_improved (F1={v4i_f1:.4f})')
    ax.fill(angles, v4i_vals, alpha=0.1, color='#31a354')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(LABELS, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('카테고리별 F1 레이더 차트\nSimple vs v3_2 vs v4_improved', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=10)
ax.grid(color='gray', alpha=0.3)

plt.tight_layout()
out_path = OUT / 'compare_radar_f1_v4_improved_20260526.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {out_path}')

In [ ]:
# 셀 6: [그래프 5] 버전별 성능 추이 (v1_2 → v2_2 → v3_2 → v4_improved)

V1_2_SIMPLE_F1 = 0.8450
V1_2_KC_F1     = 0.5310
V2_2_SIMPLE_F1 = 0.7919
V2_2_KC_F1     = 0.8076
V3_2_SIMPLE_F1 = s_f1
V3_2_KC_F1     = v3_f1

if v4i is None:
    versions    = ['v1_2\n(소규모)', 'v2_2\n(695행)', 'v3_2\n(15948행)']
    simple_hist = [V1_2_SIMPLE_F1, V2_2_SIMPLE_F1, V3_2_SIMPLE_F1]
    kc_hist     = [V1_2_KC_F1,     V2_2_KC_F1,     V3_2_KC_F1]
    kc_label    = 'KcELECTRA 파인튜닝'
else:
    versions    = ['v1_2\n(소규모)', 'v2_2\n(695행)', 'v3_2\n(15948행)', 'v4_improved\n(16512행)']
    simple_hist = [V1_2_SIMPLE_F1, V2_2_SIMPLE_F1, V3_2_SIMPLE_F1, s_f1]
    kc_hist     = [V1_2_KC_F1,     V2_2_KC_F1,     V3_2_KC_F1,     v4i_f1]
    kc_label    = 'KcELECTRA 파인튜닝'

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(versions, simple_hist, 'o-', color='#6baed6', linewidth=2.5,
        markersize=9, label='Simple (TF-IDF + LogReg)')
ax.plot(versions, kc_hist, 's-', color='#e6550d', linewidth=2.5,
        markersize=9, label=kc_label)

# 마지막 점만 녹색으로 강조 (v4_improved)
if v4i is not None:
    ax.plot(versions[-1], kc_hist[-1], '^', color='#31a354', markersize=14, zorder=5,
            label=f'v4_improved = {v4i_f1:.4f}')

for i, (s, k) in enumerate(zip(simple_hist, kc_hist)):
    ax.annotate(f'{s:.4f}', (i, s), textcoords='offset points',
                xytext=(-22, 8), fontsize=9, color='#2171b5')
    ax.annotate(f'{k:.4f}', (i, k), textcoords='offset points',
                xytext=(5, -16), fontsize=9, color='#a63603')

# v1 데이터 부족 주석
ax.annotate('데이터 부족\n(학습 실패)',
            xy=(0, V1_2_KC_F1), xytext=(0.3, V1_2_KC_F1 - 0.10),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.2),
            fontsize=9, color='gray', ha='center')

# v4_improved 최고 성능 주석
if v4i is not None and v4i_f1 > V3_2_KC_F1:
    ax.annotate('최고 성능!',
                xy=(len(versions)-1, v4i_f1),
                xytext=(len(versions)-1.5, v4i_f1 + 0.035),
                arrowprops=dict(arrowstyle='->', color='#31a354', lw=1.5),
                fontsize=11, color='#31a354', fontweight='bold')

ax.set_ylim(0.4, 1.05)
ax.set_ylabel('Macro F1', fontsize=12)
ax.set_title('버전별 성능 추이 (v1_2 → v2_2 → v3_2 → v4_improved)\n데이터·모델·학습 기법 변화에 따른 전체 이력',
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
out_path = OUT / 'compare_version_trend_v4_improved_20260526.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {out_path}')

In [ ]:
# 셀 7: [그래프 6] Precision / Recall / F1 종합 비교

if v4i is None:
    metrics   = ['Precision', 'Recall', 'F1']
    s_vals_p  = [simple['macro_precision'], simple['macro_recall'],  simple['macro_f1']]
    v3_vals_p = [v3_2['macro_precision'],   v3_2['macro_recall'],    v3_2['macro_f1']]
    x = np.arange(len(metrics))
    w = 0.3
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - w/2, s_vals_p,  w, label='Simple (베이스라인)', color='#6baed6')
    ax.bar(x + w/2, v3_vals_p, w, label='KcELECTRA v3_2', color='#e6550d')
    for i, (s, v) in enumerate(zip(s_vals_p, v3_vals_p)):
        ax.text(i - w/2, s + 0.008, f'{s:.4f}', ha='center', va='bottom', fontsize=10)
        ax.text(i + w/2, v + 0.008, f'{v:.4f}', ha='center', va='bottom', fontsize=10)
else:
    metrics    = ['Precision', 'Recall', 'F1']
    s_vals_p   = [simple['macro_precision'],  simple['macro_recall'],  simple['macro_f1']]
    v3_vals_p  = [v3_2['macro_precision'],    v3_2['macro_recall'],    v3_2['macro_f1']]
    v4i_vals_p = [v4i['macro_precision'],     v4i['macro_recall'],     v4i['macro_f1']]
    x = np.arange(len(metrics))
    w = 0.25
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w,  s_vals_p,   w, label='Simple (베이스라인)', color='#6baed6')
    ax.bar(x,      v3_vals_p,  w, label='KcELECTRA v3_2', color='#e6550d')
    ax.bar(x + w,  v4i_vals_p, w, label='KcELECTRA v4_improved', color='#31a354')
    for i, (s, v3, v4) in enumerate(zip(s_vals_p, v3_vals_p, v4i_vals_p)):
        ax.text(i - w,  s + 0.008,  f'{s:.4f}',  ha='center', va='bottom', fontsize=9)
        ax.text(i,      v3 + 0.008, f'{v3:.4f}', ha='center', va='bottom', fontsize=9)
        ax.text(i + w,  v4 + 0.008, f'{v4:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=13)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score (Macro avg)', fontsize=12)
ax.set_title('Precision / Recall / F1 종합 비교', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
out_path = OUT / 'compare_precision_recall_f1_v4_improved_20260526.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'저장: {out_path}')

In [ ]:
# 셀 8: 최종 요약 출력

print('=' * 70)
print('  최종 성능 비교 요약 (test 1,593건 — 동일 test set)')
print('  KcELECTRA: beomi/kcelectra-base (이준범)')
print('=' * 70)

if v4i:
    print(f'  {"지표":15s} {"Simple":>10s} {"v3_2":>10s} {"v4_improved":>13s} {"v3_2→v4i":>10s}')
    print('  ' + '-' * 63)
    rows = [
        ('Macro Precision', simple['macro_precision'], v3_2['macro_precision'], v4i['macro_precision']),
        ('Macro Recall',    simple['macro_recall'],    v3_2['macro_recall'],    v4i['macro_recall']),
        ('Macro F1',        simple['macro_f1'],        v3_2['macro_f1'],        v4i['macro_f1']),
    ]
    for name, s, v3, v4 in rows:
        d1 = v3 - s
        d2 = v4 - v3
        m1 = '★' if d1 >= 0.05 else 'up' if d1 > 0 else 'dn'
        m2 = '★' if d2 >= 0.005 else 'up' if d2 > 0 else 'dn'
        print(f'  {name:15s} {s:>10.4f} {v3:>10.4f} {v4:>13.4f} {d2:>+9.4f} {m2}')

    print('  ' + '-' * 63)
    print(f'\n  {"카테고리":10s} {"Simple":>8s} {"v3_2":>8s} {"v4_improved":>12s} {"v3→v4i":>8s}')
    print('  ' + '-' * 52)
    for lbl in LABELS:
        s  = simple['per_class'][lbl]['f1']
        v3 = v3_2['per_class'][lbl]['f1']
        v4 = v4i['per_class'][lbl]['f1']
        d  = v4 - v3
        sup = v4i['per_class'][lbl]['support']
        m   = 'up' if d > 0.005 else 'dn' if d < -0.005 else '~'
        print(f'  {lbl:10s} {s:>8.4f} {v3:>8.4f} {v4:>12.4f} {d:>+7.4f} {m}  (test {int(sup)}건)')

    print('\n' + '=' * 70)
    delta_vs_simple = v4i_f1 - s_f1
    delta_vs_v3     = v4i_f1 - v3_f1
    print(f'  Simple 대비:  {delta_vs_simple:+.4f}  ({"5%+ 달성 ★" if delta_vs_simple >= 0.05 else "향상"})')
    print(f'  v3_2 대비:    {delta_vs_v3:+.4f}  ({"향상 ★" if delta_vs_v3 > 0 else "동일 수준" if abs(delta_vs_v3) < 0.001 else "하락"})')
    print('=' * 70)
else:
    print(f'  Simple v3_2   Macro F1: {s_f1:.4f}')
    print(f'  KcELECTRA v3_2 Macro F1: {v3_f1:.4f}  (Simple 대비 {v3_f1-s_f1:+.4f})')
    print(f'  KcELECTRA v4_improved: 미완료 — Colab 학습 필요')
    print('=' * 70)

## 해석 가이드

### 1. 메인 비교 (Simple vs v4_improved)
- **목적**: "왜 딥러닝(트랜스포머)을 써야 하는가?" 에 대한 수치적 답변
- **기준**: Macro F1 5%+ 향상 → 트랜스포머 도입 당위성 확보

### 2. 탐색 과정 추이 (v1 → v2 → v3_2 → v4_improved)
- **v1_2**: 데이터 부족(클래스당 32개) → 학습 실패 (F1=0.5310)
- **v2_2**: 695행 → 첫 성공, Simple 근소 우위 (F1=0.8076)
- **v3_2**: 15,948행 → 5%+ 달성, 채택 (F1=0.8374)
- **v4_improved**: 학교알리미 공공데이터 증강 + 학습 기법 개선 → 목표 0.84+

### 3. v4_improved 개선 근거
| 개선 항목 | 이유 |
|----------|------|
| val/test purity | v4 실패 핵심 원인 제거 — val에 증강 데이터 혼입 시 val F1 과대평가 |
| LR=3e-5 + cosine | 더 빠른 수렴, 국소 최적점 탈출 |
| LabelSmoothingCE | 기타↔건강·안전처럼 경계 모호한 클래스에서 과신 방지 |
| BS=32 + patience=7 | 안정적 gradient + 더 철저한 탐색 |